# 00 · Colab setup and correctness checks

Keep this notebook under `notebooks/colab/`. In Google Drive, upload the project folder so the layout is:

- `My Drive/polypssl/code/polypssl/train.py`
- `My Drive/polypssl/code/polypssl/polypssl/`
- `My Drive/polypssl/data_zips/TrainDataset.zip` and `TestDataset.zip` (optional if setup downloads them)

The folder is copied to `/content/polypssl`; the notebook also accepts a ZIP as a fallback. Data goes to `/content/data`;
output paths are `/content/outputs` and new output folders are backed by `My Drive/polypssl/outputs` for checkpoint
persistence. Existing local output folders are preserved and reused. Re-running does not delete code or data, and
valid archives/checkpoints are reused.
Do not start training unless all data, weight, and test checks below pass.

In [ ]:
import os
import subprocess
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

PLATFORM = "colab"
WORK = Path("/content")
CODE_DIR = WORK / "polypssl"
CODE_SOURCE = Path("/content/drive/MyDrive/polypssl/code/polypssl")
CODE_ZIP = Path("/content/drive/MyDrive/polypssl/code/polypssl.zip")

def extract_code_archive(archive, destination):
    with zipfile.ZipFile(archive) as zf:
        names = [PurePosixPath(info.filename.replace("\\", "/")) for info in zf.infolist() if not info.is_dir()]
        if any(path.is_absolute() or ".." in path.parts or
               (path.parts and path.parts[0].endswith(":")) for path in names):
            raise ValueError(f"Unsafe path found in {archive}")
        prefix = PurePosixPath("polypssl")
        if not any(path == prefix / "train.py" for path in names):
            prefix = PurePosixPath(".")
        if not any(path == prefix / "train.py" for path in names):
            raise FileNotFoundError("The code ZIP must contain train.py and the polypssl package")
        for info in zf.infolist():
            path = PurePosixPath(info.filename.replace("\\", "/"))
            if info.is_dir() or "__MACOSX" in path.parts:
                continue
            try:
                relative = path.relative_to(prefix) if str(prefix) != "." else path
            except ValueError:
                continue
            if not relative.parts:
                continue
            target = destination.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

if (CODE_SOURCE / "train.py").is_file() and (CODE_SOURCE / "polypssl").is_dir():
    shutil.copytree(CODE_SOURCE, CODE_DIR, dirs_exist_ok=True)
    print(f"Copied project folder from Drive: {CODE_SOURCE}")
elif CODE_ZIP.is_file():
    extract_code_archive(CODE_ZIP, CODE_DIR)
    print(f"Extracted project ZIP from Drive: {CODE_ZIP}")
else:
    raise FileNotFoundError(
        "Project not found. Upload the polypssl folder to My Drive/polypssl/code/polypssl "
        "or place polypssl.zip in My Drive/polypssl/code.")
os.environ["POLYPSSL_ENV"] = PLATFORM
os.environ["POLYPSSL_DATA"] = "/content/data"
OUTPUT_DIR = Path("/content/outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/polypssl/outputs")
if not OUTPUT_DIR.exists() and not OUTPUT_DIR.is_symlink():
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.symlink_to(DRIVE_OUTPUT_DIR, target_is_directory=True)
elif OUTPUT_DIR.is_symlink() and OUTPUT_DIR.resolve() != DRIVE_OUTPUT_DIR.resolve():
    raise RuntimeError(f"Refusing unexpected outputs symlink: {OUTPUT_DIR} -> {OUTPUT_DIR.resolve()}")
elif not OUTPUT_DIR.is_symlink():
    raise RuntimeError(f"{OUTPUT_DIR} is a normal folder, so results would NOT be saved to Drive. "
                       "Use Runtime > Disconnect and delete runtime, then run this notebook again.")
os.environ["POLYPSSL_OUT"] = str(OUTPUT_DIR)

if not (CODE_DIR / "train.py").is_file():
    raise FileNotFoundError(f"train.py was not prepared in {CODE_DIR}")
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print(f"[{PLATFORM}] code: {CODE_DIR}")
print(f"[{PLATFORM}] data: {os.environ['POLYPSSL_DATA']}")
print(f"[{PLATFORM}] outputs: {os.environ['POLYPSSL_OUT']}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Install/check datasets and checkpoints

The setup inspects each archive and validates matching image/mask stems before extraction. It fails on missing, extra, or duplicate files. PolypGen and BKAI are optional; their external layouts are not part of the core preflight.

In [ ]:
subprocess.run([sys.executable, "setup_env.py"], check=True)

## Unit tests

In [ ]:
from polypssl.paths import get_locations
os.environ["POLYPSSL_TEST_WEIGHTS"] = str(get_locations().weights)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

## Label splits

In [ ]:
!ls splits

## Optional GPU speed checks

In [ ]:
!python speed_test.py --config configs/stageA/fulnet_c_mscan_full.yaml --iterations 50
!python speed_test.py --config configs/stageA/polyp_pvt_reproduce.yaml --iterations 50
!python speed_test.py --config configs/stageA/fulnet_c_pvt_full.yaml --iterations 50